# **History aware bot**

In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [2]:
db_path = "db/chroma_db"
embedding_model_path = "../../../Data/HuggingFaceEmbeddings Models/all-mpnet-base-v2"

In [3]:
# Loading embedding model

embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_path,
    model_kwargs={"device" : "cpu"},
    encode_kwargs={"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
# COnnect to DB

# since here Chroma(...) is the constructor for opening or creating a vector store, we passs embedding model as function to use when retreiving
db = Chroma(
    persist_directory=db_path,
    embedding_function=embedding_model
)

### **LLM**

In [6]:
llm = ChatOllama(model="llama3")

In [7]:
# Store chat coversations as msgs
chat_sitory = []

In [17]:
def ask_question(user_question, intent):

    # Question updating
    if chat_sitory:

        # prompt for question updating
        question_prompt= [
            SystemMessage(content="Given the chat history, rewrite the new question to be standalone and searchable. Just return the rewritten question.")
        ] + chat_sitory + [
            HumanMessage(content=f"New question : {user_question}")
        ]

        # new question genration
        print("#### new question genrating...")
        updated_question = llm.invoke(question_prompt).content.strip()

    else :
        updated_question = user_question

    # Find relevant docs for updated questoin
    retreiver = db.as_retriever(search_kwargs={"k":3})
    relevant_docs = retreiver.invoke(updated_question)
    print("Relevant docs found")

    # combined all relevant docs as pointwise single string
    combined_docs = "\n".join([f" - {doc.page_content}" for doc in relevant_docs])
    
    final_msg = f"""
        User message
        {user_question}

        Intent:
        {intent}
        
        Documents:
        {combined_docs}

        Instruction:
        Generate a helpful banking response based on the User message, Intent & Documents. If Intent is 'unknown' dont care about it. 
        If you can't find helpfull answer in docs say I'm wasn't able to find any information please contact support.
    """

    final_prompt = [
        SystemMessage(content="You are a helpful assistant that answer questions based on provided documents and conversation hostory. Answer should be less thatn 200 chars.")
    ]+ chat_sitory + [
        HumanMessage(content=final_msg)
    ]

    print("Generating answer...")
    response = llm.invoke(final_prompt).content.strip()

    print("-"*100)
    print(f"### Response: {response} ***")
    print("-"*100)

    chat_sitory.append(HumanMessage(content=user_question))
    chat_sitory.append(AIMessage(content=response))

    return response

In [18]:
def start_chat():
    print("Ask me question : type 'quit' to exit")
    intent = "unknown"
    
    while True:
        question= input("AsK : ")
        if question.lower() == 'quit':
            print("Thank you")
            break
        ask_question(question, intent)

In [19]:
chat_sitory=[]
start_chat()

Ask me question : type 'quit' to exit


AsK :  what is this bnk


Relevant docs found
Generating answer...
----------------------------------------------------------------------------------------------------
### Response: This refers to the Bank's Board Nominations and Governance Committee (BNGC), which oversees nominations and appointments of Directors, including Executive Directors. The committee ensures a transparent process, assesses candidates based on their ability to bring fresh perspectives, and follows regulatory requirements for listing. ***
----------------------------------------------------------------------------------------------------


AsK :  quit


Thank you
